In [1]:
import os

os.getcwd()

'/content'

In [3]:
!pip install wget

  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9686 sha256=770663b44787f07fa4c36a994ed3c52ee023bf6ffc04a49358cb8463738f70c0
  Stored in directory: /root/.cache/pip/wheels/8a/b8/04/0c88fb22489b0c049bee4e977c5689c7fe597d6c4b0e7d0b6a
Successfully built wget


In [5]:
import wget

if not os.path.exists("shakespeare.txt"):
    wget.download(
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    )

In [7]:
with open("input.txt", "r") as f:
    raw_text = f.read()

In [8]:
all_dialogues = raw_text.split("\n\n")

In [9]:
from pprint import pprint

pprint(all_dialogues[:10])

['First Citizen:\nBefore we proceed any further, hear me speak.',
 'All:\nSpeak, speak.',
 'First Citizen:\nYou are all resolved rather to die than to famish?',
 'All:\nResolved. resolved.',
 'First Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.',
 "All:\nWe know't, we know't.",
 'First Citizen:\n'
 "Let us kill him, and we'll have corn at our own price.\n"
 "Is't a verdict?",
 "All:\nNo more talking on't; let it be done: away, away!",
 'Second Citizen:\nOne word, good citizens.',
 'First Citizen:\n'
 'We are accounted poor citizens, the patricians good.\n'
 'What authority surfeits on would relieve us: if they\n'
 'would yield us but the superfluity, while it were\n'
 'wholesome, we might guess they relieved us humanely;\n'
 'but they think we are too dear: the leanness that\n'
 'afflicts us, the object of our misery, is as an\n'
 'inventory to particularise their abundance; our\n'
 'sufferance is a gain to them Let us revenge this with\n'
 'our pikes, ere we be

In [10]:
for dialogue in all_dialogues[:10]:
    print(dialogue)

First Citizen:
Before we proceed any further, hear me speak.
All:
Speak, speak.
First Citizen:
You are all resolved rather to die than to famish?
All:
Resolved. resolved.
First Citizen:
First, you know Caius Marcius is chief enemy to the people.
All:
We know't, we know't.
First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?
All:
No more talking on't; let it be done: away, away!
Second Citizen:
One word, good citizens.
First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.


In [11]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [12]:
def tokenize(s):
    return word_tokenize(s)

In [14]:
from typing import Optional, Tuple, Any, List
import torch
import torch.nn as nn
from dataclasses import dataclass

import numpy as np
import torch
import torchvision
from torch import nn, optim
from torchvision import transforms
import sklearn
from sklearn.metrics import confusion_matrix
import tqdm
import copy
from torch.utils.data import DataLoader, Subset

torch.autograd.set_detect_anomaly(True)
device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)
import os

num_workers = min(4, os.cpu_count())
import logging

# Configure logging to write directly to a file
logging.basicConfig(
    filename="vit.log",
    filemode="a",  # 'a' appends to the file, 'w' overwrites it on every run
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)


@dataclass(init=True)
class TrainResult:
    """
    A collection containing everything we need to know about the training results
    """

    train_losses: List[float]
    train_accs: List[float]
    val_accs: List[float]
    val_losses: List[float]


result = TrainResult(train_losses=[], train_accs=[], val_accs=[], val_losses=[])


class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0.0
        self.avg = 0.0
        self.sum = 0.0
        self.count = 0.0

    def update(self, val: float, n: int = 1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count if self.count > 0 else 0.0

    def calculate(self) -> float:
        return self.avg


def print_variance(name: str, data: torch.Tensor):
    # Compute variance across features/neurons and average across the batch
    neuron_variance = torch.mean(torch.var(data.detach().float(), dim=-1))
    print(f"{name}: Variance = {neuron_variance.item():.6f}")


class MultiHeadedAttention(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.n_hidden = n_hidden
        self.scale = n_hidden**-0.5

        self.qkv_projection = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)
        self.w_o = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, _ = x.shape

        # Shape: (B, T, 3 * num_heads * n_hidden) -> (B, num_heads, T, 3 * n_hidden)
        qkv = (
            self.qkv_projection(x)
            .reshape(B, T, self.num_heads, 3 * self.n_hidden)
            .transpose(1, 2)
        )
        q, k, v = qkv.chunk(3, dim=-1)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale

        if attn_mask is not None:
            if attn_mask.dim() == 3:
                attn_mask = attn_mask.unsqueeze(1)
            scores = scores.masked_fill(attn_mask == 0, float("-inf"))

        attn_weights = torch.softmax(scores, dim=-1)

        context = torch.matmul(attn_weights, v)
        context = context.transpose(1, 2).reshape(B, T, self.num_heads * self.n_hidden)

        output = self.w_o(context)
        return output, attn_weights


class AttentionResidual(nn.Module):
    def __init__(self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int):
        super().__init__()
        self.attn = MultiHeadedAttention(dim, attn_dim, num_heads)

        # LayerNorm applied inside the FFN sequence only
        self.ffn = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, dim),
        )

    def forward(
        self, x: torch.Tensor, attn_mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # Attention block with residual connection (no norm)
        attn_out, alphas = self.attn(x, attn_mask=attn_mask)
        x = x + attn_out

        # FFN block with residual connection (norm is first layer inside self.ffn)
        x = x + self.ffn(x)
        return x, alphas


class Transformer(nn.Module):
    def __init__(
        self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int, num_layers: int
    ):
        super().__init__()
        self.layers = nn.ModuleList(
            [
                AttentionResidual(dim, attn_dim, mlp_dim, num_heads)
                for _ in range(num_layers)
            ]
        )

    def forward(
        self,
        x: torch.Tensor,
        attn_mask: Optional[torch.Tensor] = None,
        return_attn: bool = False,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        collected_attns = []

        for layer in self.layers:
            x, alphas = layer(x, attn_mask=attn_mask)
            if return_attn:
                collected_attns.append(alphas)

        if return_attn:
            return x, torch.stack(collected_attns, dim=1)
        return x, None


class PatchEmbed(nn.Module):
    def __init__(self, img_size: int, patch_size: int, nin: int, nout: int):
        super().__init__()
        assert (
            img_size % patch_size == 0
        ), "Image dimensions must be divisible by patch size."

        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2

        self.proj = nn.Conv2d(
            in_channels=nin,
            out_channels=nout,
            kernel_size=patch_size,
            stride=patch_size,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Output: (B, num_patches, nout) in float
        return self.proj(x).flatten(2).transpose(1, 2)


class VisionTransformer(nn.Module):
    def __init__(
        self,
        n_channels: int,
        nout: int,
        img_size: int,
        patch_size: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
        num_global_tokens: int = 1,  # Number of global/CLS tokens
    ):
        super().__init__()
        self.num_global_tokens = num_global_tokens

        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, nin=n_channels, nout=dim
        )
        num_patches = self.patch_embed.num_patches

        # Learnable global/CLS tokens of shape (1, num_global_tokens, dim)

        self.cls_tokens = nn.Parameter(torch.zeros(1, num_global_tokens, dim))

        # Position embeddings covering both global tokens and image patches
        total_seq_len = num_global_tokens + num_patches
        self.pos_embed = nn.Parameter(torch.zeros(1, total_seq_len, dim))

        nn.init.trunc_normal_(self.cls_tokens, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )

        # Classification / Projection Head
        self.head = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, nout),
        )

    def forward(
        self, img: torch.Tensor, return_attn: bool = False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        B = img.shape[0]

        # 1. Patch embeddings: (B, num_patches, dim)
        embs = self.patch_embed(img)

        # 2. Expand global tokens across batch: (B, num_global_tokens, dim)
        cls_tokens = self.cls_tokens.expand(B, -1, -1)

        # 3. Concatenate global tokens with patch embeddings: (B, num_global_tokens + num_patches, dim)
        x = torch.cat((cls_tokens, embs), dim=1)

        # 4. Add position embeddings
        x = x + self.pos_embed

        # 5. Transformer forward pass
        x, alphas = self.transformer(x, attn_mask=None, return_attn=return_attn)

        # 6. Extract representations of all global tokens: (B, num_global_tokens, dim)
        global_repr = x[:, : self.num_global_tokens]

        # Aggregate global tokens:
        # - Option A: Use the primary token (index 0) if acting as a single CLS token with register tokens
        # out = self.head(global_repr[:, 0])
        #
        # - Option B: Mean-pool across all global tokens
        out = self.head(global_repr.mean(dim=1))  # (B, nout)

        return out, alphas


# evaluate the model
@torch.compile
def evaluate_cifar_model(model, criterion, val_loader):
    is_train = model.training
    model.eval()
    with torch.no_grad():
        loss_meter, acc_meter = AverageMeter(), AverageMeter()
        for img, labels in val_loader:
            # move all img, labels to device (cuda)
            img = img.to(device)
            labels = labels.to(device)
            outputs, _ = model(img)
            loss_meter.update(criterion(outputs, labels).item(), len(img))
            acc = (outputs.argmax(-1) == labels).float().mean().item()
            acc_meter.update(acc, len(img))
    model.train(is_train)
    return loss_meter.calculate(), acc_meter.calculate()


result = TrainResult(train_losses=[], train_accs=[], val_losses=[], val_accs=[])


def main():
    model = VisionTransformer(
        n_channels=3,
        nout=10,
        img_size=32,
        patch_size=4,
        dim=128,
        attn_dim=64,
        mlp_dim=128,
        num_heads=3,
        num_layers=6,
    ).to(device)
    print(model)
    MEAN = [0.4914, 0.4822, 0.4465]
    STD = [0.247, 0.2435, 0.2616]
    img_transform = transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize(mean=MEAN, std=STD)]
    )
    inv_transform = transforms.Compose(
        [
            transforms.Normalize(mean=[0.0, 0.0, 0.0], std=1 / np.array(STD)),
            transforms.Normalize(mean=-np.array(MEAN), std=[1.0, 1.0, 1.0]),
            transforms.ToPILImage(),
        ]
    )
    train_dataset = torchvision.datasets.CIFAR10(
        train=True, root="data", transform=img_transform, download=True
    )
    val_dataset = torchvision.datasets.CIFAR10(
        train=False, root="data", transform=img_transform
    )
    train_dataloader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=256,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )
    val_dataloader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=256,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )
    criterion = nn.CrossEntropyLoss()
    NUM_EPOCHS = 10
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.003)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NUM_EPOCHS, eta_min=1e-05
    )
    # result = TrainResult(train_losses=[], train_accs=[], val_losses=[], val_accs=[])
    for epoch in range(NUM_EPOCHS):
        loss_meter = AverageMeter()
        acc_meter = AverageMeter()
        for img, labels in tqdm.tqdm(
            train_dataloader, desc="Training at " + str(epoch)
        ):
            img, labels = (img.to(device), labels.to(device))
            optimizer.zero_grad()
            outputs, _ = model(img)
            loss = criterion(outputs, labels)
            loss_meter.update(loss.item(), len(img))
            acc = (outputs.argmax(-1) == labels).float().mean().item()
            acc_meter.update(acc, len(img))
            loss.backward()
            optimizer.step()
        scheduler.step()
        result.train_losses.append(loss_meter.calculate())
        result.train_accs.append(acc_meter.calculate())
        print(
            f"Train Epoch: {epoch}, Loss: {loss_meter.calculate()}, Acc: {acc_meter.calculate()}"
        )
        val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
        result.val_losses.append(val_loss)
        result.val_accs.append(val_acc)
        print(f"Val Epoch: {epoch}, Loss: {val_loss}, Acc: {val_acc}")
        best_models = "Vision_transformer_Best_" + str(epoch + 1) + ".pt"
        best_save_path = os.path.join(checkpoint_dir, best_models)
        model_path = "Vision_transformer_" + str(epoch + 1) + ".pt"
        save_path = os.path.join(checkpoint_dir, model_path)

        # todo add an early stopping criteria and restore best weights

        if epoch > 0 and result.val_losses[epoch] > result.val_losses[epoch - 1]:
            torch.save(model.state_dict(), best_save_path)
            val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
            print(f"Val Epoch: {epoch + 1}, Loss: {val_loss}, Acc: {val_acc}")
            print("Finished Training")

        else:
            torch.save(model.state_dict(), save_path)
        val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
        clean_memory_cache()
        print(f"Val Epoch: {epoch + 1}, Loss: {val_loss}, Acc: {val_acc}")
    print("Finished Training")

In [223]:
device = "cpu"

In [224]:
device

'cpu'

In [225]:
import gc
import torch


def clean_memory_cache():
    """Safely flushes CPU and GPU memory cache pools across all platforms."""
    # 1. System.gc() equivalent: Cleans host/unified reference counts
    gc.collect()

    # 2. Safely check and flush NVIDIA CUDA cache
    if torch.cuda.is_available():
        print(f"{torch.cuda.memory_allocated() / (1024 * 1024):.2f} MB")

        torch.cuda.empty_cache()

    # 3. Safely check for MPS cache support without crashing
    elif torch.backends.mps.is_available():
        mps_mem = torch.mps.driver_allocated_memory() / (1024 * 1024)
        print(f"MPS Alloc: {mps_mem:.2f} MB")
        if hasattr(torch.mps, "empty_cache"):
            try:
                torch.mps.empty_cache()
            except Exception:
                # Silently bypass if the MPS backend refuses the call
                pass

In [226]:
def tokenize(s):
    return word_tokenize(s)


class MyTokenizer:
    def __init__(self, raw_text: str):
        # raw_text     contains the text from which we will build our vocabulary

        self.start = "<START>"  # token that starts every example
        self.pad = "<PAD>"  # token used to pad examples to the same length
        self.unk = "<UNK>"  # token used if encountering a word not in our vocabulary

        vocab = np.unique(tokenize(raw_text))
        vocab = np.concatenate([np.array([self.start, self.pad, self.unk]), vocab])

        self.vocab = vocab  # array of tokens in order
        self.tok_to_id = {w: i for i, w in enumerate(vocab)}  # mapping of token to ID
        self.id_to_token = {i: w for i, w in enumerate(vocab)}
        self.vocab_size = len(self.vocab)  # size of vocabulary

    def __len__(self):
        return self.vocab_size

    def encode(self, s: str) -> torch.Tensor:
        # s           input string
        #
        # Output
        # id_tensor   a tensor of token ids, starting with the start token.t

        id_tensor = torch.from_numpy(
            np.array(
                [self.tok_to_id[self.start]]
                + [self.tok_to_id[w] for w in tokenize(s) if w in self.tok_to_id],
                dtype=np.int32,
            )
        )

        # TODO: tokenize the input using word_tokenize. Return a tensor  of the token ids, starting with the token id for the start token.
        # ============ ANSWER START ===========
        # encoded_string = tokenize(s)
        # token_ids =
        # token_ids.append(self.tok_to_id[self.start])
        # token_ids.extend(
        #     [self.tok_to_id[w] for w in encoded_string if w in self.tok_to_id.keys()]
        # )
        # id_tensor = np.array(token_ids)

        # id_tensor = torch.from_numpy(id_tensor)
        # ============ ANSWER END =============

        return id_tensor

    def decode(self, toks: torch.Tensor) -> str:
        # toks         a list of token ids
        #
        # Output
        # decoded_str  the token ids decoded back into a string (join with a space)

        # TODO: convert the token ids back to the actual corresponding words.
        # Join the tokens with a space and return the full string
        # ============ ANSWER START ===========
        return " ".join(
            [
                self.id_to_token[int(token)]
                for token in toks
                if token in self.tok_to_id.values()
            ]
        ).rstrip()

        # ============ ANSWER END =============

        # return decoded_str

    def pad_examples(self, tok_list: List[torch.Tensor]) -> torch.Tensor:
        # Pads the tensors to the right with the pad token so that they are the same length.
        #
        # tok_list       a list of tensors containing token ids (maybe of different lengths)
        #
        # Output
        # padded_tokens  shape: (len(tok_list), max length within tok_list)
        return torch.nn.utils.rnn.pad_sequence(
            tok_list, batch_first=True, padding_value=self.tok_to_id[self.pad]
        )


tok = MyTokenizer(raw_text)

In [227]:
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [228]:
len(tok)

14213

In [229]:
all_dialogues[:2]

['First Citizen:\nBefore we proceed any further, hear me speak.',
 'All:\nSpeak, speak.']

In [230]:
tok.tok_to_id[tok.start]

0

In [231]:
tok.id_to_token[1986]

np.str_('Ovid')

In [232]:
tok.id_to_token[0]

np.str_('<START>')

In [233]:
np.array([str(tok.id_to_token[0]), "Hi Deven"])

array(['<START>', 'Hi Deven'], dtype='<U8')

In [234]:
# tokenizer test cases
input_string = "KING RICHARD III:\nSay that I did all this for love of her. bluye"
enc = tok.encode(input_string)
print(enc)

# for x in enc:
#     print(x, type(x), x in tok.tok_to_id.values(), x in tok.id_to_token.values())
dec = tok.decode(enc)
print(dec)
# print("<START> KING RICHARD III : Say that I did all this for love of her .")
assert dec == "<START> KING RICHARD III : Say that I did all this for love of her ."

tensor([    0,  1593,  2182,  1481,   223,  2343, 12742,  1476,  5704,  3319,
        12795,  6848,  8727,  9608,  7655,   221], dtype=torch.int32)
<START> KING RICHARD III : Say that I did all this for love of her .


In [235]:
# enc = tok.encode(all_dialogues[])

# Part 4.B

### Debug

In [236]:
class DialogueDataset:
    def __init__(self, tokenizer: MyTokenizer, lines: List[str], max_N: int):
        # tokenizer    an instance of MyTokenizer
        # lines        a list of strings. each element in an example in the dataset
        # max_N        the maximum number of tokens allowed per example. More than this will be truncated
        self.lines = lines
        self.tokenizer = tokenizer
        self.max_N = max_N

    def __len__(self) -> int:
        return len(self.lines)

    # def __iter__(self):
    #     for line in self.lines:
    #         yield self.tokenizer.encode(line)[: self.max_N]

    def __getitem__(self, idx: int) -> torch.Tensor:
        # returns the example at int encoded by the tokenizer
        # truncates the example if it is more than max_N tokens
        return self.tokenizer.encode(self.lines[idx])[: self.max_N]

    def __getitems__(self, indices: int):
        return [self.__getitem__(idx) for idx in indices]


ds = DialogueDataset(tok, all_dialogues, max_N=200)

In [237]:
def collate_fn(examples: List[torch.Tensor]):
    """
    # examples        a batch of tensors containing token ids (maybe of different lengths)
    # Outputs a dictionary containing
    #   input_ids     a single tensor with all of the examples padded (from the right) to the max
    #                 length within the batch. shape:(B, max length within examples)
    #   input_mask    a tensor indicating which tokens are padding and should be ignored. 0 if padding
    #                 and 1 if not. shape: (B, max length within examples)
    """
    new_input_ids = tok.pad_examples(examples)
    attn_mask = torch.ones(new_input_ids.shape)  # 1s should not be ignored

    # causal attention mask
    attn_mask[new_input_ids == tok.tok_to_id[tok.pad]] = (
        0  # should be ignored if it is a padded
    )
    return {"input_ids": tok.pad_examples(examples), "input_mask": attn_mask}

In [238]:
first = ds[2]
second = ds[1]
# token_list = torch.tensor([first,second])
tokens = [first, second]
torch.nn.utils.rnn.pad_sequence(tokens, padding_value=tok.tok_to_id[tok.pad])

tensor([[    0,     0],
        [ 1151,   323],
        [  708,   223],
        [  223,  2517],
        [ 3050,   219],
        [ 3506, 12008],
        [ 3319,   221],
        [10983,     1],
        [10724,     1],
        [12921,     1],
        [ 5706,     1],
        [12733,     1],
        [12921,     1],
        [ 6507,     1],
        [  225,     1]], dtype=torch.int32)

In [239]:
d = collate_fn(tokens)

In [33]:
d["input_ids"]

tensor([[    0,  1151,   708,   223,  3050,  3506,  3319, 10983, 10724, 12921,
          5706, 12733, 12921,  6507,   225],
        [    0,   323,   223,  2517,   219, 12008,   221,     1,     1,     1,
             1,     1,     1,     1,     1]], dtype=torch.int32)

In [34]:
d["input_mask"]

tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [35]:
clean_memory_cache()

0.00 MB


In [36]:
from torch.utils.data import DataLoader, Subset

In [240]:
training_dl = torch.utils.data.DataLoader(
    ds, batch_size=64, collate_fn=collate_fn
)  # collate function is just a dictionary..

In [241]:
input_id, input_mask = next(iter(training_dl)).items()

In [242]:
a, b = (next(iter(training_dl))).items()

In [243]:
x1, x2 = a

In [244]:
print(x1)
x2

input_ids


tensor([[   0, 1151,  708,  ...,    1,    1,    1],
        [   0,  323,  223,  ...,    1,    1,    1],
        [   0, 1151,  708,  ...,    1,    1,    1],
        ...,
        [   0, 1733,  223,  ...,    1,    1,    1],
        [   0, 1151, 2385,  ...,    1,    1,    1],
        [   0, 1733,  223,  ...,    1,    1,    1]], dtype=torch.int32)

In [245]:
# take a look at an example of an element from the training dataloader
from IPython.display import display

for batch in training_dl:
    print(batch.keys())
    print(batch["input_ids"])
    break

dict_keys(['input_ids', 'input_mask'])
tensor([[   0, 1151,  708,  ...,    1,    1,    1],
        [   0,  323,  223,  ...,    1,    1,    1],
        [   0, 1151,  708,  ...,    1,    1,    1],
        ...,
        [   0, 1733,  223,  ...,    1,    1,    1],
        [   0, 1151, 2385,  ...,    1,    1,    1],
        [   0, 1733,  223,  ...,    1,    1,    1]], dtype=torch.int32)


## Part 4.C

In [246]:
embs = torch.ones((32, 100, 128))
B, T, _ = embs.shape
pos_ids = torch.arange(T).expand(B, -1)
print(f"{pos_ids.shape=}")
pos_E = nn.Embedding(200, 128)
print(pos_E)
x = pos_E(pos_ids)

pos_ids.shape=torch.Size([32, 100])
Embedding(200, 128)


In [247]:
print(f"{B=},{T=}")

B=32,T=100


In [248]:
pos_E

Embedding(200, 128)

In [249]:
a = torch.arange(T).expand(size=(B, -1))
embedding_layer = nn.Embedding(100, 128)
x = embedding_layer(a)

In [250]:
x = x.detach()

In [251]:
x.shape

torch.Size([32, 100, 128])

In [252]:
torch.tensor(10).repeat(2, 1, 2).shape

torch.Size([2, 1, 2])

In [253]:
torch.ones(B, B).unsqueeze(dim=0).repeat(1, 1, 1).shape

torch.Size([1, 32, 32])

In [254]:
x.shape

torch.Size([32, 100, 128])

In [ ]:
import dataclasses
from dataclasses import dataclass
from functools import wraps


from typing import List

import numpy as np
import torch
import torchvision
import tqdm
from torch import nn, optim
from torchvision import transforms


@dataclass(init=True)
class TrainResult:
    """
    A collection containing everything we need to know about the training results
    """

    train_losses: List[float]
    train_accs: List[float]
    val_accs: List[float]
    val_losses: List[float]


device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)


class AverageMeter:
    def __init__(self):
        self.num = 0
        self.tot = 0

    def update(self, val: float, sz: float):
        self.num += val * sz
        self.tot += sz

    def calculate(self) -> float:
        return self.num / self.tot


def averager(func):
    num = 0
    total = 0

    def update(val, sz):
        nonlocal num
        nonlocal total
        num += val * sz
        total += sz

    @wraps(func)
    def inner():
        nonlocal num, total
        update(num / total)

    return inner


def print_variance(name, data):
    neuron_variance = torch.mean(torch.var(data, dim=0))
    print(f"name={name!r}, Variance={neuron_variance.float()}")


class MultiAttentionHead(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        super().__init__()
        self.H = num_heads
        self.n_hidden = n_hidden
        self.qkv = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)
        self.WO = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask=None
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
                attention mask
                masks future inputs

        that those blank tokens get filled in with the prediction

        """
        batch_size, token_size, dim = x.shape
        QKV = self.qkv(x)
        QKV = QKV.reshape(batch_size, token_size, self.H * self.n_hidden * 3)
        QKV = QKV.transpose(-2, -1)
        Q, K, V = QKV.chunk(3, dim=-1)
        A = Q @ K.transpose(-2, -1) / self.n_hidden**0.5
        scores = A
        if attn_mask is not None:
            "\n            causal masking future inputs of 0 to negative so that softmax will return 0\n"
            scores = scores.masked_fill(attn_mask.unsqueeze(1) == 0, float("-inf"))
        A = torch.softmax(scores, dim=-1)
        Z = A @ V
        Z = Z.transpose(-2, -1)
        Z = Z.reshape(batch_size, token_size, self.H * self.n_hidden)
        Z = self.WO(Z)
        return (Z, A)


import torch
from torch import nn


class MultiHeadedAttention(nn.Module):
    def __init__(self, dim: int, n_hidden: int, num_heads: int):
        super().__init__()
        self.dim = dim
        self.num_heads = num_heads
        self.n_hidden = n_hidden
        self.qkv_projection = nn.Linear(dim, num_heads * n_hidden * 3, bias=False)
        self.W0 = nn.Linear(num_heads * n_hidden, dim)

    def forward(
        self, x: torch.Tensor, attn_mask=None
    ) -> tuple[torch.Tensor, torch.Tensor]:
        B, T, dim = x.shape
        qkv = self.qkv_projection(x)
        qkv = qkv.reshape(B, T, self.num_heads, 3 * self.n_hidden).transpose(1, 2)
        q, k, v = qkv.chunk(3, dim=-1)
        scores = torch.matmul(q, k.transpose(-2, -1)) / self.n_hidden**0.5
        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask.unsqueeze(1) == 0, float("-inf"))
        attn_alphas = torch.softmax(scores, dim=-1)
        context = attn_alphas @ v
        A = context.transpose(1, 2).reshape(B, T, self.num_heads * self.n_hidden)
        Z = self.W0(A)
        return (Z, attn_alphas)


class AttentionResidual(nn.Module):
    def __init__(self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int):
        super().__init__()
        self.attn = MultiHeadedAttention(dim, attn_dim, num_heads)
        self.norm1 = nn.LayerNorm(dim)
        self.ffn = nn.Sequential(
            nn.Linear(dim, mlp_dim), nn.GELU(), nn.Linear(mlp_dim, dim)
        )
        self.norm2 = nn.LayerNorm(dim)

    def forward(
        self, x: torch.Tensor, attn_mask=False
    ) -> tuple[torch.Tensor, torch.Tensor]:
        print_variance("Input", x)
        Z, A = self.attn(x=self.norm1(x), attn_mask=attn_mask)
        print_variance("after first attention block output", Z)
        x = Z + x
        print_variance("after residual 1 adding output to Z", x)
        ffn_out = self.ffn(self.norm2(x))
        x = ffn_out + x
        print_variance("after Applying FeedForward output with residual", x)
        print("Residual Block\n\n")
        return (x, A)


class Transformer(nn.Module):
    def __init__(
        self, dim: int, attn_dim: int, mlp_dim: int, num_heads: int, num_layers: int
    ):
        super().__init__()
        self.layers = nn.ModuleList(
            [
                AttentionResidual(dim, attn_dim, mlp_dim, num_heads)
                for _ in range(num_layers)
            ]
        )

    def forward(
        self, x: torch.Tensor, attn_mask=False, return_attn=False
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        """
        # x                the inputs. shape: (B x T x dim)
        # attn_mask        an attention mask. Pass this to each of the AttentionResidual layers!
        #                  shape: (B x T x T)
        #
        # Outputs:
        # attn_output      shape: (B x T x dim)
        # attn_alphas      If return_attn is False, return None. Otherwise return the attention weights
        #                  of each of each of the attention heads for each of the layers.
        #                  shape: (B x Num_layers x Num_heads x T x T)
        """
        output = None
        collected_attns = []
        Z = None
        A = []
        print(f"\n\n-------------------------Transformer: {x.shape}")
        for residual in self.layers:
            x, alphas = residual(x, attn_mask=attn_mask)
            if return_attn is not None:
                A.append(alphas)
        if return_attn:
            return (x, torch.stack(A, dim=1))
        else:
            return (x, None)


class PatchEmbed(nn.Module):
    """Image to Patch Embedding"""

    def __init__(self, img_size: int, patch_size: int, nin: int, nout: int):
        super().__init__()
        assert img_size % patch_size == 0
        self.img_size = img_size
        self.flattened_patch = (img_size // patch_size) ** 2
        self.patch_embedding = nn.Conv2d(
            in_channels=nin,
            out_channels=nout,
            kernel_size=patch_size,
            stride=patch_size,
        )
        self.nout = nout

    def forward(self, x: torch.Tensor):
        """TODO: Implement the patch embedding. You want to split up the image into square patches of the given patch size. Then each patch_size x patch_size square should be linearly projected into an embedding of size nout. Hint: Take a look at nn.Conv2d. How can this be used to perform the patch embedding?"""
        out = self.patch_embedding(x)
        out = out.flatten(2)
        out = out.transpose(1, 2).long()
        print(f"Output shape in patch embedding: out.shape={out.shape!r}")
        return out


class VisionTransformer(nn.Module):
    def __init__(
        self,
        n_channels: int,
        nout: int,
        img_size: int,
        patch_size: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
    ):
        super().__init__()
        self.patch_embed = PatchEmbed(
            img_size=img_size, patch_size=patch_size, nin=n_channels, nout=dim
        )
        self.pos_E = nn.Embedding((img_size // patch_size) ** 2, dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, nout))

    def forward(
        self, img: torch.Tensor, return_attn=False
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        embs = self.patch_embed(img)
        B, T, _ = embs.shape
        print(f"embs.shape={embs.shape!r}")
        pos_ids = torch.arange(T).expand(B, -1).to(embs.long().to(device))
        embs += self.pos_E(pos_ids).long()
        print_variance("Embeddings", embs.float())
        cls_token = self.cls_token.expand(len(embs), -1, -1)
        x = torch.cat([cls_token, embs], dim=1)
        x, alphas = self.transformer(x, attn_mask=None, return_attn=return_attn)
        print_variance("Output", x)
        out = self.head(x)[:, 0]
        print_variance("Projected Output", x)
        return (out, alphas)


@torch.compile
def evaluate_cifar_model(model, criterion, val_loader):
    is_train = model.training
    model.eval()
    with torch.no_grad():
        loss_meter, acc_meter = (AverageMeter(), AverageMeter())
        for img, labels in val_loader:
            img = img.to(device)
            labels = labels.to(device)
            outputs, _ = model(img)
            loss_meter.update(criterion(outputs, labels).item(), len(img))
            acc = (outputs.argmax(-1) == labels).float().mean().item()
            acc_meter.update(acc, len(img))
    model.train(is_train)
    return (loss_meter.calculate(), acc_meter.calculate())


def main():
    model = VisionTransformer(
        n_channels=3,
        nout=10,
        img_size=32,
        patch_size=4,
        dim=128,
        attn_dim=64,
        mlp_dim=128,
        num_heads=3,
        num_layers=6,
    ).to(device)
    MEAN = [0.4914, 0.4822, 0.4465]
    STD = [0.247, 0.2435, 0.2616]
    img_transform = transforms.Compose(
        [transforms.ToTensor(), transforms.Normalize(mean=MEAN, std=STD)]
    )
    inv_transform = transforms.Compose(
        [
            transforms.Normalize(mean=[0.0, 0.0, 0.0], std=1 / np.array(STD)),
            transforms.Normalize(mean=-np.array(MEAN), std=[1.0, 1.0, 1.0]),
            transforms.ToPILImage(),
        ]
    )
    train_dataset = torchvision.datasets.CIFAR10(
        train=True, root="data", transform=img_transform, download=True
    )
    val_dataset = torchvision.datasets.CIFAR10(
        train=False, root="data", transform=img_transform
    )
    train_dataloader = torch.utils.data.DataLoader(
        train_dataset, batch_size=256, shuffle=True, num_workers=2
    )
    val_dataloader = torch.utils.data.DataLoader(
        val_dataset, batch_size=256, shuffle=False, num_workers=2
    )
    criterion = nn.CrossEntropyLoss()
    NUM_EPOCHS = 10
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.003)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    result = TrainResult(train_losses=[], train_accs=[], val_losses=[], val_accs=[])
    for epoch in range(NUM_EPOCHS):
        loss_meter = AverageMeter()
        acc_meter = AverageMeter()
        for img, labels in tqdm.tqdm(
            train_dataloader, desc="Training at " + str(epoch)
        ):
            img, labels = (img.to(device), labels.to(device))
            optimizer.zero_grad()
            outputs, _ = model(img)
            loss = criterion(outputs, labels)
            loss_meter.update(loss.item(), len(img))
            acc = (outputs.argmax(-1) == labels).float().mean().item()
            acc_meter.update(acc, len(img))
            loss.backward()
            optimizer.step()
        scheduler.step()
        result.train_losses.append(loss_meter.calculate())
        result.train_accs.append(acc_meter.calculate())
        print(
            f"Train Epoch: {epoch}, Loss: {loss_meter.calculate()}, Acc: {acc_meter.calculate()}"
        )
        val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
        result.val_losses.append(val_loss)
        result.val_accs.append(val_acc)
        print(f"Val Epoch: {epoch}, Loss: {val_loss}, Acc: {val_acc}")
        if epoch > 0 and result.val_losses[epoch] > result.val_losses[epoch - 1]:
            torch.save(
                model.state_dict(), "Vision_transformer_Best_" + str(epoch + 1) + ".pt"
            )
            val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
            print(f"Val Epoch: {epoch + 1}, Loss: {val_loss}, Acc: {val_acc}")
            print("Finished Training")
            break
        else:
            torch.save(
                model.state_dict(), "Vision_transformer_" + str(epoch + 1) + ".pt"
            )
        val_loss, val_acc = evaluate_cifar_model(model, criterion, val_dataloader)
        print(f"Val Epoch: {epoch + 1}, Loss: {val_loss}, Acc: {val_acc}")
        print("Finished Training")

In [256]:
assert (torch.tril(torch.ones(B, B)).unsqueeze(dim=0).repeat(100, 1, 1)).shape == (
    T,
    B,
    B,
)

In [257]:
device = "cpu"

In [258]:
import torch
import torch.nn as nn
from typing import Tuple, Optional


class DialogueGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        max_N: int,
        dim: int,
        attn_dim: int,
        mlp_dim: int,
        num_heads: int,
        num_layers: int,
    ):
        super().__init__()

        # Token embeddings track vocabulary words
        self.token_embeddings = nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=dim
        )

        # Positional embeddings track sequence sequence length (up to max_N)
        self.pos_embeddings = nn.Embedding(num_embeddings=max_N, embedding_dim=dim)

        self.transformer = Transformer(
            dim=dim,
            attn_dim=attn_dim,
            mlp_dim=mlp_dim,
            num_heads=num_heads,
            num_layers=num_layers,
        )

        # Projection Head
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, vocab_size))

    def forward(
        self, input_ids: torch.Tensor, return_attn=False
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        B, T = input_ids.shape
        device = input_ids.device

        # FIX 1: Generate positional indices (0, 1, ..., T-1) on the correct device
        pos_ids = torch.arange(0, T, dtype=torch.long, device=device).unsqueeze(
            0
        )  # Shape: (1, T)

        # Retrieve embeddings correctly
        embs = self.token_embeddings(input_ids) + self.pos_embeddings(pos_ids)

        # Create the causal attention mask dynamically matching the tensor device
        causal_attn_mask = (
            torch.tril(torch.ones(T, T, device=device)).unsqueeze(0).repeat(B, 1, 1)
        ).to(
            device
        )  # Shape: (B, T, T)

        x, alphas = self.transformer(
            embs, attn_mask=causal_attn_mask, return_attn=return_attn
        )
        out = self.head(x)
        return out, alphas

    def generate(self, input_ids, num_tokens):
        # Assume batch size 1
        with torch.no_grad():
            for i in range(num_tokens):
                # FIX 2: Ensure we don't pass a sequence longer than max_N to forward
                # Standard GPT slice window to keep tracking the last tokens within context window
                if input_ids.shape[1] > self.pos_embeddings.num_embeddings:
                    input_ids_window = input_ids[
                        :, -self.pos_embeddings.num_embeddings :
                    ]
                else:
                    input_ids_window = input_ids

                out, _ = self.forward(input_ids_window)
                new_token = torch.argmax(out[:, [-1]], -1)
                input_ids = torch.cat([input_ids, new_token], dim=1)
        return input_ids

In [259]:
device = "cpu"

In [260]:
len(tok)

14213

In [261]:
model = DialogueGPT(
    vocab_size=len(tok),
    max_N=200,
    dim=128,
    attn_dim=64,
    mlp_dim=128,
    num_heads=3,
    num_layers=6,
)

In [262]:
model

DialogueGPT(
  (token_embeddings): Embedding(14213, 128)
  (pos_embeddings): Embedding(200, 128)
  (transformer): Transformer(
    (layers): ModuleList(
      (0-5): 6 x AttentionResidual(
        (attn): MultiHeadedAttention(
          (qkv_projection): Linear(in_features=128, out_features=576, bias=False)
          (w_o): Linear(in_features=192, out_features=128, bias=True)
        )
        (ffn): Sequential(
          (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=128, out_features=128, bias=True)
          (2): GELU(approximate='none')
          (3): Linear(in_features=128, out_features=128, bias=True)
        )
      )
    )
  )
  (head): Sequential(
    (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=128, out_features=14213, bias=True)
  )
)

In [263]:
len(tok)

14213

In [264]:
batch = next(iter(training_dl))

In [265]:
batch

{'input_ids': tensor([[   0, 1151,  708,  ...,    1,    1,    1],
         [   0,  323,  223,  ...,    1,    1,    1],
         [   0, 1151,  708,  ...,    1,    1,    1],
         ...,
         [   0, 1733,  223,  ...,    1,    1,    1],
         [   0, 1151, 2385,  ...,    1,    1,    1],
         [   0, 1733,  223,  ...,    1,    1,    1]], dtype=torch.int32),
 'input_mask': tensor([[1., 1., 1.,  ..., 0., 0., 0.],
         [1., 1., 1.,  ..., 0., 0., 0.],
         [1., 1., 1.,  ..., 0., 0., 0.],
         ...,
         [1., 1., 1.,  ..., 0., 0., 0.],
         [1., 1., 1.,  ..., 0., 0., 0.],
         [1., 1., 1.,  ..., 0., 0., 0.]])}

In [266]:
batch.keys()

dict_keys(['input_ids', 'input_mask'])

In [267]:
device = "cpu"

In [268]:
model = model.to("cpu")
x = batch["input_ids"]
mask = batch["input_mask"]
x = x.to("cpu")
model(x)

(tensor([[[ 0.5544,  0.1092, -0.0019,  ..., -0.1643,  0.5718,  0.6538],
          [-0.6945,  0.6205,  1.0550,  ...,  0.2554,  0.1234,  0.7188],
          [ 0.4177, -0.6461,  0.2916,  ...,  1.0604, -1.2040,  0.4605],
          ...,
          [ 0.0939, -0.0091,  0.0138,  ...,  0.4074, -0.4196,  1.1996],
          [ 0.2413, -0.6292, -0.2112,  ...,  0.1094, -0.1013,  0.1737],
          [ 0.0490, -0.3011,  0.6004,  ..., -0.7021,  0.3793,  0.7269]],
 
         [[ 0.5544,  0.1092, -0.0019,  ..., -0.1643,  0.5718,  0.6538],
          [ 0.1895,  0.0508,  0.2154,  ..., -0.0031, -0.0935,  0.3048],
          [ 0.0184,  0.0266,  0.1678,  ...,  0.3249, -0.9424, -0.3292],
          ...,
          [ 0.0938, -0.0056, -0.0077,  ...,  0.3898, -0.4483,  1.2045],
          [ 0.2290, -0.6167, -0.2254,  ...,  0.1051, -0.1336,  0.1793],
          [ 0.0503, -0.3013,  0.5793,  ..., -0.7003,  0.3417,  0.7335]],
 
         [[ 0.5544,  0.1092, -0.0019,  ..., -0.1643,  0.5718,  0.6538],
          [-0.6945,  0.6205,

In [269]:
CUDA_LAUNCH_BLOCKING = 1

In [270]:
import os

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [271]:
mask

tensor([[1., 1., 1.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.],
        ...,
        [1., 1., 1.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.]])

In [272]:
mask = mask.to(device)
x * mask

tensor([[   0., 1151.,  708.,  ...,    0.,    0.,    0.],
        [   0.,  323.,  223.,  ...,    0.,    0.,    0.],
        [   0., 1151.,  708.,  ...,    0.,    0.,    0.],
        ...,
        [   0., 1733.,  223.,  ...,    0.,    0.,    0.],
        [   0., 1151., 2385.,  ...,    0.,    0.,    0.],
        [   0., 1733.,  223.,  ...,    0.,    0.,    0.]])

## Part 4.D

In [206]:
a = torch.tensor([1, 2, 3])
a = a[:-1]
a

tensor([1, 2])

In [ ]:
clean_memory_cache()

In [273]:
device = "cpu"

In [274]:
import torch
import torch.nn as nn


class DialogueLoss(nn.Module):
    def __init__(self):
        super().__init__()
        # Explicitly ignore padding index if your mask allows it,
        # but since we manual-mask below, reduction="none" is perfect.
        self.criterion = nn.CrossEntropyLoss(reduction="none")

    def forward(
        self, logits: torch.Tensor, input_ids: torch.Tensor, inp_mask: torch.Tensor
    ):
        # Shift logits and targets for next-token prediction
        relevant_logits = logits[:, :-1, :]  # Shape: (B, T-1, V)
        relevant_tokens = input_ids[:, 1:]  # Shape: (B, T-1)
        shift_mask = inp_mask[:, 1:]  # Shape: (B, T-1)

        # PyTorch CrossEntropyLoss expects vocabulary dimension as the second dimension: (B, V, T-1)
        relevant_logits = relevant_logits.permute(0, 2, 1)

        # Compute token-wise loss
        loss = self.criterion(relevant_logits, relevant_tokens)

        # FIX: Ensure mask matches the loss tensor's data type AND device exactly
        shift_mask = shift_mask.to(dtype=loss.dtype, device=loss.device)

        # Apply mask to nullify losses on padding tokens
        loss = loss * shift_mask

        # Avoid division by zero smoothly using clamp
        return torch.sum(loss) / torch.clamp(torch.sum(shift_mask), min=1e-9)

In [275]:
loss = nn.CrossEntropyLoss(reduction="none")
# Outputs a tensor of the same size as the batch (or spatial dimensions for tasks like segmentation), giving you full control over how to weight, filter, or apply custom masks to individual samples

In [276]:
inp = torch.randn(2, 2, requires_grad=True)
target = torch.randn(2, 2)

In [277]:
all_dialogues[0]

'First Citizen:\nBefore we proceed any further, hear me speak.'

In [278]:
target.sum(axis=1)

tensor([0.9805, 0.1269])

In [279]:
loss(inp, target)

tensor([-0.1864,  0.1123], grad_fn=<NegBackward0>)

In [280]:
logits = torch.randn(3, 5, requires_grad=True)
targets = torch.randn(3, 5)
x = loss(logits, targets)

In [281]:
logits

tensor([[-1.0669,  1.0821, -1.3339, -0.2943,  1.5774],
        [ 0.2780, -1.1595, -0.8421, -0.7698,  0.6739],
        [-0.8083,  1.9323,  0.7554,  0.0481, -1.4364]], requires_grad=True)

In [282]:
target

tensor([[-0.5514,  1.5318],
        [-0.0060,  0.1328]])

## Part 4.F

In [283]:
import torch.optim as optim

model = DialogueGPT(
    vocab_size=tok.vocab_size,
    max_N=200,
    dim=128,
    attn_dim=64,
    mlp_dim=128,
    num_heads=3,
    num_layers=6,
)
criterion = DialogueLoss()

NUM_EPOCHS = 1

optimizer = optim.AdamW(
    model.parameters(), lr=0.0001, weight_decay=0
)  # implement in homework
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

In [284]:
model

DialogueGPT(
  (token_embeddings): Embedding(14213, 128)
  (pos_embeddings): Embedding(200, 128)
  (transformer): Transformer(
    (layers): ModuleList(
      (0-5): 6 x AttentionResidual(
        (attn): MultiHeadedAttention(
          (qkv_projection): Linear(in_features=128, out_features=576, bias=False)
          (w_o): Linear(in_features=192, out_features=128, bias=True)
        )
        (ffn): Sequential(
          (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (1): Linear(in_features=128, out_features=128, bias=True)
          (2): GELU(approximate='none')
          (3): Linear(in_features=128, out_features=128, bias=True)
        )
      )
    )
  )
  (head): Sequential(
    (0): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (1): Linear(in_features=128, out_features=14213, bias=True)
  )
)

In [294]:
for p, m in model.named_parameters():
    print(m.device)

cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu
cpu


In [295]:
from torch.utils.data import Subset

In [219]:
training_dl.dataset

In [182]:
dl = Subset(training_dl.dataset, torch.arange(3))

In [183]:
# from torch.utils.data import DataLoader
# mini_dl = DataLoader(dl, batch_size=training_dl.batch_size)

In [296]:
len(training_dl)

113

In [297]:
device = "cpu"

In [ ]:
# Time estimate: around 30 minutes on T4 GPU
# Training
import tqdm

for epoch in range(NUM_EPOCHS):  # loop over the dataset multiple times
    loss_meter = AverageMeter()
    for inp_dict in tqdm.tqdm(training_dl):
        # get the inputs; data is a list of [inputs, labels]
        inp_ids, inp_mask = inp_dict["input_ids"], inp_dict["input_mask"]
        print(f"{inp_ids.shape=}")
        inp_ids = inp_ids.to(device)
        inp_mask = inp_mask.to(device)
        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs, _ = model(input_ids=inp_ids)
        print(f"{outputs.shape=}")
        loss = criterion(outputs, inp_ids, inp_mask)
        loss_meter.update(loss.item(), len(inp_dict["input_ids"]))
        loss.backward()
        optimizer.step()
    scheduler.step()

    # print example
    inp = tok.encode("").unsqueeze(0).to(device)
    print(tok.decode(model.generate(inp, 10)[0].cpu()))

    print(
        f"Train Epoch: {epoch}, Loss: {loss_meter.calculate():0.4f}, LR: {scheduler.get_last_lr()[0]}"
    )

  0%|          | 0/113 [00:00<?, ?it/s]

inp_ids.shape=torch.Size([64, 200])


In [ ]:
x